# 01 — Explore & Clean

MiAdvisor · WolfHacks 2026

This notebook prepares the WolfHacks synthetic student-outcomes dataset for modeling:

1. Load the **Student Data** sheet
2. Inspect shape, target balance and missing values
3. Make the stratified 80/20 train/test split
4. Impute missing values (using training rows only)
5. One-hot encode the categorical columns
6. Build the final feature matrix and pick the model features
7. Export `cleaned_data.csv` and `feature_config.json` for the training notebook and the ML service

> All records are synthetic. No real student PII is used.

## 1. Load the data

The workbook has three sheets (`README`, `Student Data`, `Data Dictionary`). Only `Student Data` holds records.
Each row is one student in one course. The target is `at_risk`: **1** means the student finished with a D or F (final score below 70), **0** means C or better.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "../data/wolfhacks_2026_student_outcomes.xlsx"
TARGET = "at_risk"
ID_COL = "student_id"
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_excel(DATA_PATH, sheet_name="Student Data")
df.head()

,student_id,course,class_year,credit_hours,work_hours_per_week,avg_weekly_study_hours,study_sessions_logged,materials_uploaded,practice_quizzes_taken,avg_practice_quiz_score,flashcards_reviewed,avg_days_started_before_exam,on_time_submission_rate,missed_deadlines,late_night_study_pct,attendance_rate,avg_sleep_hours,midterm_score,at_risk
0,S1000,BIO 181,Sophomore,14,3.0,0.5,10,7,6,70.8,135,1.5,0.83,1,0.28,0.83,7.4,65,0
1,S1001,BIO 181,Freshman,16,11.0,2.9,16,10,8,77.3,49,4.0,0.81,3,0.07,0.84,6.8,69,0
2,S1002,MA 241,Sophomore,13,NaN,1.9,24,9,9,52.9,124,0.0,0.77,2,0.41,0.80,6.5,56,0
3,S1003,CSC 216,Sophomore,15,13.0,2.8,28,18,30,96.3,326,4.0,0.88,3,0.27,0.95,6.4,98,0
4,S1004,EC 201,Junior,16,17.0,4.9,54,14,13,69.5,129,5.5,0.94,0,0.22,0.81,6.0,76,0


## 2. Shape, target distribution and nulls

We check:
- **Shape**: how many rows and columns we have.
- **Target distribution**: roughly 28% of students are at risk. Because the classes are imbalanced, a model that always guesses "not at risk" already gets about 72% accuracy, so the training notebook must beat that baseline on **recall**, not accuracy.
- **Null counts**: which columns need imputation.

In [2]:
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

print("Target distribution (at_risk):")
print(pd.DataFrame({
    "count": df[TARGET].value_counts(),
    "share": df[TARGET].value_counts(normalize=True).round(4),
}))

nulls = df.isnull().sum()
print("\nNull counts (columns with any missing values):")
print(nulls[nulls > 0])

Shape: 800 rows x 19 columns

Target distribution (at_risk):
         count   share
at_risk               
0          578  0.7225
1          222  0.2775

Null counts (columns with any missing values):
work_hours_per_week        24
avg_practice_quiz_score    40
avg_sleep_hours            40
dtype: int64


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
credit_hours,800.0,14.788750,1.811277,12.00,13.00,15.00,16.00,18.00
work_hours_per_week,776.0,6.525773,8.566814,0.00,0.00,2.00,12.00,35.00
avg_weekly_study_hours,800.0,4.295500,2.303728,0.50,2.70,4.20,5.70,11.40
study_sessions_logged,800.0,40.570000,22.888861,3.00,23.00,40.00,55.00,105.00
materials_uploaded,800.0,8.257500,4.793999,0.00,5.00,8.00,12.00,22.00
practice_quizzes_taken,800.0,13.640000,7.802981,0.00,8.00,13.00,19.00,41.00
avg_practice_quiz_score,760.0,74.864211,10.400990,40.00,67.70,74.60,81.40,100.00
flashcards_reviewed,800.0,191.183750,117.667985,0.00,103.75,187.00,271.00,639.00
avg_days_started_before_exam,800.0,3.131000,2.276679,0.00,1.20,3.00,4.60,12.10
on_time_submission_rate,800.0,0.831000,0.096700,0.51,0.77,0.83,0.90,1.00


## 3. Train/test split (before any imputation)

We split **before** computing anything from the data. If the imputation medians were computed on all 800 rows, a little information from the test set would leak into training, and the test score would be slightly optimistic.

- 80% train / 20% test
- **Stratified** on `at_risk`, so both sides keep the ~28% at-risk rate
- `random_state=42` for reproducibility

The split is saved as a `split` column in `cleaned_data.csv`, so the training notebook reuses exactly these rows.

In [4]:
train_ids, test_ids = train_test_split(
    df[ID_COL], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df[TARGET]
)
df["split"] = np.where(df[ID_COL].isin(train_ids), "train", "test")
is_train = df["split"] == "train"

print(df.groupby("split")[TARGET].agg(rows="size", at_risk_rate="mean").round(4))

       rows  at_risk_rate
split                    
test    160        0.2750
train   640        0.2781


## 4. Handle missing values

| Column | Missing | Strategy | Why |
|---|---|---|---|
| `work_hours_per_week` | 24 of 800 (self-reported) | **median** of training rows | Robust to skew; a few students working 30+ hours would pull the mean up |
| `avg_sleep_hours` | 40 of 800 (self-reported) | **median** of training rows | Same reasoning; keeps the typical value |
| `avg_practice_quiz_score` | 40 of 800 (no quizzes taken) | **0** | No quiz taken means no signal. The blanks line up exactly with `practice_quizzes_taken == 0` (checked below), so this isn't random missingness |

The medians come from the **training rows only** and are then applied to both train and test. We store them in `feature_config.json` so the ML service fills missing inputs the same way at prediction time.

In [5]:
# Sanity check: quiz score is blank exactly when no quizzes were taken
print(pd.crosstab(
    df["avg_practice_quiz_score"].isna().rename("quiz_score_missing"),
    (df["practice_quizzes_taken"] == 0).rename("zero_quizzes_taken"),
))

zero_quizzes_taken  False  True 
quiz_score_missing              
False                 760      0
True                    0     40


In [6]:
clean = df.copy()

imputation_values = {}
for col in ["work_hours_per_week", "avg_sleep_hours"]:
    median_val = float(clean.loc[is_train, col].median())
    clean[col] = clean[col].fillna(median_val)
    imputation_values[col] = median_val
    print(f"{col}: filled with training median = {median_val}")

clean["avg_practice_quiz_score"] = clean["avg_practice_quiz_score"].fillna(0.0)
imputation_values["avg_practice_quiz_score"] = 0.0
print("avg_practice_quiz_score: filled with 0.0")

remaining = int(clean.isnull().sum().sum())
print(f"\nRemaining nulls: {remaining}")
assert remaining == 0

work_hours_per_week: filled with training median = 2.0
avg_sleep_hours: filled with training median = 6.2
avg_practice_quiz_score: filled with 0.0

Remaining nulls: 0


## 5. One-hot encode `course` and `class_year`

Decision trees in scikit-learn need numeric inputs, so each category becomes its own 0/1 column (for example `course_MA 241`, `class_year_Freshman`).

We keep **every** category instead of dropping one (`drop_first=False`). Trees aren't affected by the redundancy that linear models worry about, and keeping all of them makes feature importances and the "why" explanations easier to read.

The training notebook reads these columns from `cleaned_data.csv` rather than re-encoding, so the encoding is defined in one place only.

In [7]:
CATEGORICAL_COLS = ["course", "class_year"]
categories = {col: sorted(clean[col].unique().tolist()) for col in CATEGORICAL_COLS}
for col, cats in categories.items():
    print(f"{col}: {cats}")

encoded = pd.get_dummies(clean, columns=CATEGORICAL_COLS, drop_first=False, dtype=int)

course: ['BIO 181', 'CH 101', 'CSC 216', 'EC 201', 'ECE 200', 'MA 241', 'PY 205', 'ST 311']
class_year: ['Freshman', 'Junior', 'Senior', 'Sophomore']


## 6. Final feature matrix and model features

`X` holds every cleaned column except the ID, the target and the split label: 27 columns, all kept in `cleaned_data.csv` for analysis.

The model itself trains on a smaller set of **11 features** (`model_features`). Each one is something MiAdvisor can collect from a student, through calendar deadlines, the study dashboard, quizzes or a quick survey, so the deployed `/predict` endpoint can always build a complete input. The left-out columns are:

- `course_*` and `class_year_*`: these describe who the student is, not what they're doing, so they give the student nothing to act on. Leaving them out also keeps a student's course or class year from becoming a reason they're flagged.
- `work_hours_per_week`: has missing values, and the student can't change it within the course.
- `credit_hours`, `materials_uploaded`, `practice_quizzes_taken`: not part of the agreed feature set. `avg_practice_quiz_score` already captures quiz performance.

In [8]:
X = encoded.drop(columns=[ID_COL, TARGET, "split"])
y = encoded[TARGET]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}\n")
print("All cleaned features:")
for i, name in enumerate(X.columns, 1):
    print(f"  {i:2d}. {name}")

Feature matrix shape: (800, 27)
Target shape: (800,)

All cleaned features:
   1. credit_hours
   2. work_hours_per_week
   3. avg_weekly_study_hours
   4. study_sessions_logged
   5. materials_uploaded
   6. practice_quizzes_taken
   7. avg_practice_quiz_score
   8. flashcards_reviewed
   9. avg_days_started_before_exam
  10. on_time_submission_rate
  11. missed_deadlines
  12. late_night_study_pct
  13. attendance_rate
  14. avg_sleep_hours
  15. midterm_score
  16. course_BIO 181
  17. course_CH 101
  18. course_CSC 216
  19. course_EC 201
  20. course_ECE 200
  21. course_MA 241
  22. course_PY 205
  23. course_ST 311
  24. class_year_Freshman
  25. class_year_Junior
  26. class_year_Senior
  27. class_year_Sophomore


In [9]:
MODEL_FEATURES = [
    "attendance_rate",
    "missed_deadlines",
    "on_time_submission_rate",
    "avg_practice_quiz_score",
    "midterm_score",
    "avg_weekly_study_hours",
    "flashcards_reviewed",
    "avg_days_started_before_exam",
    "late_night_study_pct",
    "avg_sleep_hours",
    "study_sessions_logged",
]
assert set(MODEL_FEATURES) <= set(X.columns)

print(f"Model features ({len(MODEL_FEATURES)}):")
for name in MODEL_FEATURES:
    print(f"  - {name}")
print(f"\nKept in CSV but not used by the model: {[c for c in X.columns if c not in MODEL_FEATURES]}")

Model features (11):
  - attendance_rate
  - missed_deadlines
  - on_time_submission_rate
  - avg_practice_quiz_score
  - midterm_score
  - avg_weekly_study_hours
  - flashcards_reviewed
  - avg_days_started_before_exam
  - late_night_study_pct
  - avg_sleep_hours
  - study_sessions_logged

Kept in CSV but not used by the model: ['credit_hours', 'work_hours_per_week', 'materials_uploaded', 'practice_quizzes_taken', 'course_BIO 181', 'course_CH 101', 'course_CSC 216', 'course_EC 201', 'course_ECE 200', 'course_MA 241', 'course_PY 205', 'course_ST 311', 'class_year_Freshman', 'class_year_Junior', 'class_year_Senior', 'class_year_Sophomore']


## 7. Export

- **`cleaned_data.csv`**: `student_id`, `split`, all 27 cleaned features and `at_risk`. This is the input to the training notebook.
- **`feature_config.json`**: the model's feature order, the imputation values (from training rows), category lists and split settings. The ML service uses it to turn a raw request into the exact same feature vector.

In [10]:
cleaned_df = pd.concat([encoded[[ID_COL, "split"]], X, y], axis=1)
cleaned_df.to_csv("cleaned_data.csv", index=False)

dummy_prefixes = tuple(f"{col}_" for col in CATEGORICAL_COLS)
feature_config = {
    "target": TARGET,
    "id_column": ID_COL,
    "model_features": MODEL_FEATURES,
    "all_features": X.columns.tolist(),
    "numeric_features": [c for c in X.columns if not c.startswith(dummy_prefixes)],
    "categorical_columns": categories,
    "imputation_values": imputation_values,
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": TARGET},
    "n_rows": int(len(cleaned_df)),
    "positive_rate": float(np.round(y.mean(), 4)),
}
with open("feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print(f"Saved cleaned_data.csv ({cleaned_df.shape[0]} rows x {cleaned_df.shape[1]} cols)")
print(f"Saved feature_config.json ({len(MODEL_FEATURES)} model features, {X.shape[1]} total)")

Saved cleaned_data.csv (800 rows x 30 cols)
Saved feature_config.json (11 model features, 27 total)
